# Práctica para evaluación módulo 'Large Language Models'
## RAG multi-turno con LangGraph

### Carga de API_KEY desde archivo de claves
Se cargan las API keys proporcionadas por el profesor (`GOOGLE_API_KEY` y `OPENAI_API_KEY`) de archivo .env en la raíz del proyect 

In [25]:
# Carga de variables de entorno (carga variables de /.env: GOOGLE_API_KEY y OPENAI_API_KEY)
import os
from load_dotenv import load_dotenv
load_dotenv()

True

### Elección de modelos
Dado que se plantea un RAG, se necesita un modelo para generación y otro para la generación de los embeddings que contendrán en contexto derivado de la documentación aportada
Dado que no se requieren grandes capaciades y que se está en un entorno con un capacidad de uso de tokens limitada, se escogen:
- `GENERATION_MODEL`:
    **gemini-2.5-flash-lite**: *Our most cost-efficient multimodal model, offering the fastest performance for high-frequency, lightweight tasks. Gemini 2.5 Flash-Lite is best for high-volume classification, simple data extraction, and extremely low-latency applications where budget and speed are the primary constraints.*
- `EMBEDDING_MODEL`m :
    **gemini-embedding-001**: *A specialized engine for high-dimensional vector representation, providing efficient numerical mapping of text and images. The Gemini Embedding model is best for semantic search, document retrieval, and recommendation systems that require fast, scalable similarity calculations across large datasets.*    

In [26]:
# Elección modelos
# gemini-2.5-flash-lite:
# Our most cost-efficient multimodal model, offering the fastest performance for high-frequency, lightweight tasks. Gemini 2.5 Flash-Lite is best for high-volume classification, simple data extraction, and extremely low-latency applications where budget and speed are the primary constraints.
GENERATION_MODEL = 'gemini-2.5-flash-lite'


EMBEDDING_MODEL = 'gemini-embedding-001'
# gemini-embedding-001:
# A specialized engine for high-dimensional vector representation, providing efficient numerical mapping of text and images. The Gemini Embedding model is best for semantic search, document retrieval, and recommendation systems that require fast, scalable similarity calculations across large datasets.

### Instanciar modelos
Se escoge como `LangChain` como entorno en vistas a usar LangGraph más adelante, por la propiedad que tiene de estructurar un chat (sisntema multi-turno)

En `GENERATION_MODEL` se establecen los siguientes parámetros:
- `temperature`=**0.2**. Se elige un valor bajo porqué se prima la fiabilidad del contenido a la forma. Esto es una respuesta quzá menos elaborada en términos de lenguage natural pero con mayor fiabilidad en que el contenido se ajuste a lo esperado. El motivo es que en una fase inicial del proyecto y con un llm ligero se quiere poder descartar al máximo las posibles fuentes de error. No se especifian ni top-P ni top-K porqué son hasta cierto punto redundantes con la temperatura y así no generar posibles inconsistencias en lo que el moelo entienda
- `max_output_tokens`=**150**. Se escoge un número bajo para no correr el riesgo de vaciar la cuota antes de finalizar el ejercicio. En todo caso, cuando se tenga todo funcionando, ya se aumentará para las útlimas pruebas
- `response_mime_type`: **'text/plain'**. El por defecto, texto plano. En estr punto se pretende un chat con el usuario, no redirigir a ninguna herramienta

In [48]:
# Instanciar los modelos
from langchain.chat_models import init_chat_model
from langchain_google_genai import GoogleGenerativeAIEmbeddings

llm = init_chat_model(GENERATION_MODEL,
                      model_provider="google_genai",
                      temperature=0.2,         # baja temperatura, se incentivan respuestas deterministas
                      #top_p=0.3,               # solamente toma en cuentra la sigueinte palabara más probable: puede resultar en repetición palabaras pero se asgura al ausencia de fallos
                      max_output_tokens=150,    # por no quedarse sin crédito en la práctica
                      response_mime_type='text/plain')
#embeddings = GoogleGenerativeAIEmbeddings(model=EMBEDDING_MODEL)

### Cargar documentación con el contexto
De momento se carga el pdf, tal cual, con `PyPDFLoader`. El formato de texto absolutamente plano que se obtiene no permite hacer un chunking que aproveche la estructura que proveen los títulos. Se ve que el modelo no funciona muy bien con esto, aun probando distintos tamaños y overlap. Se deja para una fase posterior cargar el pdf previamiente convertido a markdown para solucionarlo

In [49]:
# Cargar el documento
from langchain_community.document_loaders import PyPDFLoader

pages = PyPDFLoader('data/TENERIFE.pdf').load()

In [50]:
pages[0].metadata

{'producer': 'PyPDF',
 'creator': 'Microsoft Word',
 'creationdate': '2025-07-13T20:00:01+00:00',
 'title': 'Microsoft Word - TENERIFE.docx',
 'moddate': '2025-07-13T20:00:01+00:00',
 'source': 'data/TENERIFE.pdf',
 'total_pages': 25,
 'page': 0,
 'page_label': '1'}

### Chunking
Se usa `RecursiveCharacterTextSplitter` que usa la puntuación para ajustar donde corta el chunk
En el documento, se ha visto que, más o menos, el trozo mas largo de contenido simánticamente homogéneo es de unos 900 carácteres. También los hay de mucho más cortos. Se han probado varias opciones en longitud y overlap, sin conseguir que ninguna sea capaz de hacer un retrive correcto a 'cuales son las playas de la Orotava'

Para poder tracear mejor el resultado se añade en metadata un identificador del chunk y su tamaño. El nombredel documento y la página deonde se ha extraído el chunk ya los pone por defecto PyPDFLoader en 'source' y 'page_label'

In [51]:
# Chunking
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=400,
    chunk_overlap=80,
    add_start_index=True,
)

chunks = text_splitter.split_documents(pages)

In [52]:
for i, chunk in enumerate(chunks):
    chunk.metadata['chunk']=i
    chunk.metadata['len']=len(chunk.page_content)

chunks[0].metadata

{'producer': 'PyPDF',
 'creator': 'Microsoft Word',
 'creationdate': '2025-07-13T20:00:01+00:00',
 'title': 'Microsoft Word - TENERIFE.docx',
 'moddate': '2025-07-13T20:00:01+00:00',
 'source': 'data/TENERIFE.pdf',
 'total_pages': 25,
 'page': 0,
 'page_label': '1',
 'start_index': 0,
 'chunk': 0,
 'len': 324}

### Creación de la base de datos vectorial
De momento se usa `InMemoryVectorStore`, esto es, no se crea una base de datos persistente
Se intentó con `FAISS`, pero daba errores y no se consiguió qe funcionase


In [53]:
# Creación embedings, que se guardan en base de datos vectorial
from langchain_core.vectorstores import InMemoryVectorStore

vector_store = InMemoryVectorStore(embeddings)
document_ids = vector_store.add_documents(chunks)

In [54]:
# Probar el retrieval de la vector_store
question = "playas Norte"
chunks = vector_store.similarity_search(question, k=3)

for chunk in chunks:
    print(f"chunk number: {chunk.metadata['chunk']}")
    print(f"souce: {chunk.metadata['source']}")
    print(f"source page: {chunk.metadata['page_label']}")
    print(chunk.page_content)
    print(80*'=')

chunk number: 18
souce: data/TENERIFE.pdf
source page: 11
tendréis poca playa (especialmente a la del Ancón) y tened mucho respeto al mar en 
estas playas, que la corriente es muy traicionera. 
 
• Puerto de La Cruz 
Es la zona turística del Norte de Tenerife. De hecho, hace años era el núcleo turístico 
de la isla hasta que empezó a ganar mucho peso el Sur de Tenerife.
chunk number: 36
souce: data/TENERIFE.pdf
source page: 16
De esta zona os recomendaríamos las siguientes playas: 
o Playa de Torviscas
chunk number: 17
souce: data/TENERIFE.pdf
source page: 10
o La Playa de Los Patos 
o Playa del Ancón 
 
Nota: Para llegar estas playas hay que caminar un rato y algunas son de difícil acceso 
(la de Los Patos). También hay que ir a estas playas cuando la marea esté baja, si no


In [55]:
'''
# Embeddings
from langchain_google_vertexai import VertexAIEmbeddings
from langchain_community.vectorstores import FAISS

embeddings = GoogleGenerativeAIEmbeddings(model=EMBEDDING_MODEL)

try:
    vectorstore = FAISS.load_local(
        'data', 
        embeddings, 
        allow_dangerous_deserialization=True)  # Required if loading a previously saved local index
except:
    vectorstore = FAISS.from_documents( pages, embeddings)
    vectorstore.save_local('data/faiss_index')
'''

"\n# Embeddings\nfrom langchain_google_vertexai import VertexAIEmbeddings\nfrom langchain_community.vectorstores import FAISS\n\nembeddings = GoogleGenerativeAIEmbeddings(model=EMBEDDING_MODEL)\n\ntry:\n    vectorstore = FAISS.load_local(\n        'data', \n        embeddings, \n        allow_dangerous_deserialization=True)  # Required if loading a previously saved local index\nexcept:\n    vectorstore = FAISS.from_documents( pages, embeddings)\n    vectorstore.save_local('data/faiss_index')\n"

### Definición del objeto RAGState
Contiene el estado del RAG:
- histórico de respuestas
- contexto para el último mensaje de usuario

In [56]:
from typing import Annotated, Sequence
from typing_extensions import TypedDict
from langgraph.graph.message import add_messages
from langchain_core.messages import BaseMessage

class RAGState(TypedDict):
    # messages es una secuencia (lista en sentido amplio) de mensajes LangChain (BaseMessage es clase base para HUmanMessage, SystemMessage, etc)
    # con el comentario 'add_messages' se indica que la función 'add_messages' de LangGraph añada la respuesta obtenida a la lista
    messages: Annotated[Sequence[BaseMessage], add_messages]
    # context se va a actualizar en cada iteración con el retrieval asociado al último mensaje de usuario
    context: str

### Creación del prompt template con LangChain con 'placeholder' para que se actualice en cada turno con el histórico de respuestas y el mensaje de usuario
Con ello se puede encapsular de una vez las instrucciones de sistema y poder entrar como variables pregunta y contexto obtenido en la fase de 'retrieval'

En el mensaje de 'system':
- se especifica que únicamente se tenga en cuenta el contexto proporcionado por la documentación incluida en el RAG. No se quieren 'interferencias' externas, se quiere evaluar la capacidad de manejar la documentación dada
- se pide que si en el contexto no encuentra la información, que claramente lo diga, que no invente
- se pide que cite la fuente (se va a incluir un campo 'Fuente' en los metadatos de los chunk

Se cambia el tipo 'user' por 'placeholder', de forma que LangGraph va rellenar automáticamente con RAGState['messages'], que contiene el histórico de respuestas más el último mensaje de usuario en cada turno

In [57]:
# Prompt template
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.prompts import MessagesPlaceholder

make_prompt = ChatPromptTemplate.from_messages(
    [
        (
             'system', 
             'Eres un asistente para dar información sobre lugares de interés en Tenerife'
             'Responde únicamente usando este contexto:\n{context}'
             'Si el contexto no contiene la respuesta, di claramente: información no disponible en el contexto'
             'No inventes datos'
             'Cita Fuentes al final',
        ),
        MessagesPlaceholder("messages"),
    ]
)  

### Generación del string de contexto a partir de los chunk obtenidos en 'retrieval'
El contexto debe pasarse como un string al `prompt template`. Se incluyen los metadatos que interessan (para cada chunk, documento fuente, página del documento e identificador de chunk) para poder auditar la respuesta

In [58]:
# Generación de string de contexto a partir de chunks devueltos por similarity_search
def make_context_string(chunks):
    context = ""
    for chunk in chunks:
        context = context + f"Fuente: {chunk.metadata['source']},\
        Página: {chunk.metadata['page_label']},\
        chunk: {chunk.metadata['chunk']}\
        \n{chunk.page_content}\n\n"
    return context
        

### Definición de nodo retrieval
Este nodo va a actualizar RAGState['context'] con el retrieval de la base de datos vectorial

In [59]:
k = 3  # k deja de ser un parámetro 'de usuario' para ser una constante de configuración
def retrieve_node(state: RAGState):
    # el último mensaje de RAGState['messages'] es la entrada de usuario, los anteriores el histórico de respuestas
    chunks = vector_store.similarity_search(state['messages'][-1], k=k)
    # retorna el string de contexto en diccionario con la etiqueta 'context', lo que permite a LangChain lingar con RAGState['conext']
    return {'context': make_context_string(chunks)}

### Definición de nodo de generación
Crea el prompt a partir RAGState actulizado y lo entra en el llm y actualiza RAGState con la respuesta

In [60]:
def generate_node(state: RAGState):
    # se crea el prompt
    prompt = make_prompt.invoke({'messages': state['messages'], 'context': state['context']})
    # se llama al llm con el promt y se retorna la respuesta em diccionario con la etiqueta 'messages', lo que permite a LangChain lingar con RAGState['messages']
    return {'messages': llm.invoke(prompt)}

### Construcción del grafo con memoria

In [61]:
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.graph import StateGraph, START, END

# Definición del objeto grafo a partir de RAGState
workflow = StateGraph(RAGState)

# Creación de los objetos nodos del grafo
workflow.add_node('retriever', retrieve_node)
workflow.add_node('generator', generate_node)

# Ubicación de los nodos en el grafo (princpio, final y conexiones)
workflow.add_edge(START, 'retriever')
workflow.add_edge('retriever', 'generator')
workflow.add_edge('generator', END)

# Compilación del grafo con la memoria, gestionada en RAGState por InMemorySaver
graph = workflow.compile(checkpointer=InMemorySaver())

### Constucción de la función de query
Tiene como parámetros la prenguta de usuario y thread_id. thread_id es un hilo de conversación; si se quiere seguir con esa conversación se debe introducir si thread_id. k es ahora un parámetro de sistema
Para poder auditar la respuesta, la posibilidad de que además de la respuesta de también el contexto usado
Ejecuta primero el retrieval y después la llamada al modelo generativo con el contexto obtenido
Retorna un diccionario con:
- `question`: la pregunta de usuario
- `answer`: la respuesta dada por la query
- `context`: el contexto que usa el modelo generativo
- `prompt`: el prompt pasado al modelo, que incluye las instrucciones de sistema, la pregunta y el contexto

In [62]:
from langchain_core.messages import HumanMessage

def TenerifePOI(question: str, chat_id: str):
    
    graph.invoke({'messages': [HumanMessage(content=question)]},
                 config={'configurable': {'thread_id': chat_id}}
    )
    
    return {
        "question": question,
        "answer": RAGState['messages'][-1],
        "context": RAGState['context'],
    }

### Prueba de la query
Falla en la pregunta 'playas en la Orotava'. Son Bollullo, Patos y Ancona. Se piensa que esto se podría mejorar si el chunking tomase los títulos: la información tiene un sentido, de título en adelante, no hacia los dos lados; Las 3 playas están entre dos títulos, el conteniendo el primero 'Orotava'

In [63]:
res = TenerifePOI('playas en la Orotava',
                  'playas')
print(res['answer'])

GoogleGenerativeAIError: Error embedding content: file uri and mime_type are required.